In [2]:
#import relevant libraries: pip install re, pip install natsort, pip install plotly==5.10.0
import sys
import os
import glob

import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import datetime as dt8
import matplotlib.pyplot as plt
mpl.use("svg")
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Source Sans Pro'
import dabest
import seaborn as sns
import NLCLIMB 
import NLMATH

import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#NOTE: SUPPRESSES WARNINGS!

import warnings

warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

print(dabest.__version__)
print(np.version.version)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:01<00:00,  9.36it/s]


Numba compilation complete!
2025.03.27
2.1.3


In [3]:
#Initial file processing
computer1 = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
homecomp = "D:"
filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data\\Fecundity\\"
openPath = homecomp + filedir

files = os.listdir(openPath)
filename = openPath + "Fecundityfile.csv"

dfe=pd.read_csv(filename)

In [4]:
df = pd.concat([dfe['Genotype'], dfe['Taken for expt'], dfe['Pupated'] , dfe['Eclosed']], axis = 1)

if df['Genotype'].str.contains("eOPN3").any():
   df['Genotype'] = df['Genotype'].str.replace("eOPN3", "AsOPN3")


df['number'] = pd.factorize(df.Genotype)[0].tolist()

repeats = int(df[df.number == 1].number.value_counts().values[0])

In [5]:
dftotal = pd.DataFrame()
dfnumbers = pd.DataFrame()
for n in df['Genotype'].unique():
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()
    embryos = int(df[df['Genotype'] == n]['Taken for expt'].sum())
    pupated = int(df[df['Genotype'] == n]['Pupated'].sum())
    eclosed = int(df[df['Genotype'] == n]['Eclosed'].sum())
    
    #total embryos
    df1[n+" (EM)"] = [1]*embryos
    df2[n+"_embryos"] = [embryos]
    
    #total pupate
    failpupate =embryos-pupated                        
    df1[n + " (LE)"] = [0]*failpupate + [1]*pupated
    df2[n + "_pupate"] = [pupated]
    
    #total eclosed
    df1[n + " (PE)"] = [0]*failpupate + [0]*(pupated-eclosed) + [1]*eclosed 
    df2[n + "_eclose"] = [eclosed]
    
    df1['ID'] = range(1, len(df1)+1)
    # dftotal = pd.concat([dftotal,df1], axis =1).reset_index(drop = True)
    dfnumbers = pd.concat([dfnumbers,df2], axis =1).reset_index(drop = True)
    
    dfmelted = pd.melt(df1, id_vars = ["ID"], value_vars=df1.columns[:-1])

    dftotal = pd.concat([dfmelted, dftotal], axis = 0).reset_index(drop = True)
    
print(dftotal)

     ID           variable  value
0     1    elav x ACR (EM)      1
1     2    elav x ACR (EM)      1
2     3    elav x ACR (EM)      1
3     4    elav x ACR (EM)      1
4     5    elav x ACR (EM)      1
..   ..                ...    ...
541  68  w1118 x elav (PE)      1
542  69  w1118 x elav (PE)      1
543  70  w1118 x elav (PE)      1
544  71  w1118 x elav (PE)      1
545  72  w1118 x elav (PE)      1

[546 rows x 3 columns]


In [8]:
multi_groups_baseline = dabest.load(dftotal, idx=(("w1118 x elav (EM)", "w1118 x elav (LE)", 'w1118 x elav (PE)' )
                                                  ,("elav x AsOPN3 (EM)", "elav x AsOPN3 (LE)", 'elav x AsOPN3 (PE)')
                                                  ,("elav x ACR (EM)", "elav x ACR (LE)", 'elav x ACR (PE)')), x = "variable", y = "value",
                                        proportional=True, paired="baseline", id_col="ID")


f = multi_groups_baseline.mean_diff.plot(title="Fecundity plots (n = " + str(repeats) + " replicates)",
    prop_sample_counts=True, prop_sample_counts_kwargs={"color":"black"}, contrast_ylim = (-1.5, 1),  fontsize_rawxlabel=8, fontsize_rawylabel=10, fontsize_contrastxlabel=8, contrast_marker_size=4,
                                    fontsize_contrastylabel=10, fontsize_delta2label=10);

#f.savefig(openPath + "Fecundity.svg", bbox_inches = 'tight')

f

C:\Users\user\AppData\Roaming\Python\Python310\site-packages\dabest\_effsize_objects.py:523: UserWarning: The lower limit of the BCa interval of baseline curve cannot be computed. It is set to the effect size itself. All bootstrap values were likely all the same.
  warnings.warn(err_temp.substitute(lim_type="lower"), stacklevel=0)
C:\Users\user\AppData\Roaming\Python\Python310\site-packages\dabest\_effsize_objects.py:527: UserWarning: The upper limit of the BCa interval of baseline curve cannot be computed. It is set to the effect size itself. All bootstrap values were likely all the same.
  warnings.warn(err_temp.substitute(lim_type="upper"), stacklevel=0)
C:\Users\user\AppData\Roaming\Python\Python310\site-packages\dabest\_effsize_objects.py:323: UserWarning: The lower limit of the BCa interval cannot be computed. It is set to the effect size itself. All bootstrap values were likely all the same.
  warnings.warn(err_temp.substitute(lim_type="lower"), stacklevel=0)
C:\Users\user\AppDa

<Figure size 1350x600 with 2 Axes>

In [7]:
f

<Figure size 1350x600 with 2 Axes>